# Feature Engineering — Riesgo de retraso en el pago

## Objetivo del notebook

El objetivo de este notebook es transformar el dataset original de cuentas a cobrar en un dataset analítico preparado para modelado supervisado.

La unidad de análisis será una factura y el objetivo predictivo será estimar, en el momento de emisión, la probabilidad de que dicha factura sea pagada después de su fecha de vencimiento.

La variable objetivo será:

`late_payment = 1` si `SettledDate > DueDate`  
`late_payment = 0` en caso contrario.

El feature engineering se realizará respetando estrictamente la dimensión temporal. Para cada factura únicamente se utilizará información que estuviese disponible en su fecha de emisión.

## Prevención de data leakage

Para una factura emitida en una fecha `t`:

- una factura anterior se considera emitida si `InvoiceDate < t`;
- su comportamiento de pago se considera conocido únicamente si `SettledDate < t`;
- una factura anterior se considera abierta si `InvoiceDate < t` y `SettledDate >= t`.

No se utilizarán como predictores variables que contengan información posterior al momento de emisión, como:

- `SettledDate`;
- `DaysToSettle`;
- `DaysLate`;
- `days_delay_signed`;
- `days_late_calc`;
- otras variables derivadas del resultado final del pago.

`Disputed` se excluirá inicialmente del modelo por no disponer de una fecha que permita determinar cuándo se conoció la disputa.

`PaperlessDate` tampoco se utilizará directamente como predictor, ya que puede contener información posterior a la fecha de emisión de algunas facturas.

## Familias de variables a construir

El dataset final incluirá características pertenecientes a varios grupos:

1. **Características de la factura actual**
   - importe de la factura;
   - modalidad de facturación;
   - país o grupo de cliente.

2. **Variables temporales**
   - mes;
   - trimestre;
   - día de la semana;
   - posición temporal dentro del histórico.

3. **Histórico conocido del cliente**
   - número de pagos anteriores conocidos;
   - tasa histórica de retraso;
   - retraso medio, mediano y máximo;
   - comportamiento reciente de pago.

4. **Exposición del cliente en la fecha de emisión**
   - número de facturas abiertas;
   - número de facturas vencidas;
   - importe pendiente;
   - importe vencido;
   - antigüedad de la deuda vencida.

5. **Variables relativas al comportamiento habitual del cliente**
   - importe actual frente al importe medio histórico;
   - importe actual frente a la mediana histórica;
   - desviación respecto al patrón habitual del cliente.

## Resultado esperado

El resultado del notebook será un dataset Gold con una fila por factura:

`gold_invoice_risk.parquet`

Este dataset constituirá la entrada de las fases posteriores de modelado y evaluación.

La parte de simulación de tesorería se desarrollará posteriormente en un notebook independiente, combinando los cobros reales históricos con pagos, gastos y compromisos financieros sintéticos.

En esta primera fase se realizarán:

- carga de los datos originales;
- normalización de nombres y tipos;
- conversión de fechas;
- reconstrucción de la variable objetivo;
- validaciones básicas de calidad;
- ordenación temporal de las observaciones.

Las variables históricas del cliente se construirán posteriormente respetando estrictamente la información disponible en la fecha de emisión de cada factura.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


In [2]:
PROJECT_ROOT = Path("..")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
GOLD_DIR = PROJECT_ROOT / "data" / "gold"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
GOLD_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE = RAW_DIR / "accounts_receivable_ibm.csv"

print("Raw:", RAW_FILE)
print("Processed:", PROCESSED_DIR)
print("Gold:", GOLD_DIR)

Raw: ..\data\raw\accounts_receivable_ibm.csv
Processed: ..\data\processed
Gold: ..\data\gold


In [3]:
raw_df = pd.read_csv(RAW_FILE)

df = raw_df.copy()

print(f'Filas: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')

df.head()

Filas: 2466
Columnas: 12


,countryCode,customerID,PaperlessDate,invoiceNumber,InvoiceDate,DueDate,InvoiceAmount,Disputed,SettledDate,PaperlessBill,DaysToSettle,DaysLate
0,391,0379-NEVHP,4/6/2013,611365,1/2/2013,2/1/2013,55.94,No,1/15/2013,Paper,13,0
1,406,8976-AMJEO,3/3/2012,7900770,1/26/2013,2/25/2013,61.74,Yes,3/3/2013,Electronic,36,6
2,391,2820-XGXSB,1/26/2012,9231909,7/3/2013,8/2/2013,65.88,No,7/8/2013,Electronic,5,0
3,406,9322-YCTQO,4/6/2012,9888306,2/10/2013,3/12/2013,105.92,No,3/17/2013,Electronic,35,5
4,818,6627-ELFBK,11/26/2012,15752855,10/25/2012,11/24/2012,72.27,Yes,11/28/2012,Paper,34,4


In [4]:
rename_columns = {
    "countryCode": "country_code",
    "customerID": "customer_id",
    "PaperlessDate": "paperless_date",
    "invoiceNumber": "invoice_id",
    "InvoiceDate": "invoice_date",
    "DueDate": "due_date",
    "InvoiceAmount": "invoice_amount",
    "Disputed": "disputed",
    "PaperlessBill": "paperless_bill",
    "SettledDate": "settled_date",
    "DaysToSettle": "days_to_settle_original",
    "DaysLate": "days_late_original",
}

df = df.rename(columns=rename_columns)

df.columns.tolist()

['country_code',
 'customer_id',
 'paperless_date',
 'invoice_id',
 'invoice_date',
 'due_date',
 'invoice_amount',
 'disputed',
 'settled_date',
 'paperless_bill',
 'days_to_settle_original',
 'days_late_original']

## Conversión y normalización de tipos

In [5]:
# Fechas
date_columns = [
    "invoice_date",
    "due_date",
    "settled_date",
    "paperless_date",
]

for col in date_columns:
    df[col] = pd.to_datetime(
        df[col],
        format="%m/%d/%Y",
        errors="coerce"
    )

# Identificadores
df["invoice_id"] = (
    df["invoice_id"]
    .astype("string")
    .str.strip()
)

df["customer_id"] = (
    df["customer_id"]
    .astype("string")
    .str.strip()
)

# country_code -> no debe tratarse como número
df["country_code"] = (
    df["country_code"]
    .astype("string")
    .str.strip()
)

# Importe
df["invoice_amount"] = pd.to_numeric(
    df["invoice_amount"],
    errors="coerce"
)

# Variables categoricas
df["paperless_bill"] = (
    df["paperless_bill"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df["disputed"] = (
    df["disputed"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df.dtypes

country_code               string[python]
customer_id                string[python]
paperless_date             datetime64[ns]
invoice_id                 string[python]
invoice_date               datetime64[ns]
due_date                   datetime64[ns]
invoice_amount                    float64
disputed                   string[python]
settled_date               datetime64[ns]
paperless_bill             string[python]
days_to_settle_original             int64
days_late_original                  int64
dtype: object

## Reconstruccion de Target

In [6]:
# Diferencia respecto al vencimiento
df["days_delay_signed"] = (
    df["settled_date"]
    - df["due_date"]
).dt.days

# Target
df["late_payment"] = (
    df["days_delay_signed"] > 0
).astype("int8")

df["days_late"] = (
    df["days_delay_signed"]
    .clip(lower=0)
)

**no serán features del modelo.** Las mantenemos porque necesitamos conocer el resultado de las facturas antiguas para poder construir después variables como:
- `customer_previous_late_rate`
- `customer_previous_avg_delay`

## Validaciones de consistencia

In [7]:
quality_checks = pd.Series({
    "rows": len(df),

    "duplicated_rows": (
        df.duplicated().sum()
    ),

    "duplicated_invoice_id": (
        df["invoice_id"]
        .duplicated()
        .sum()
    ),

    "missing_invoice_id": (
        df["invoice_id"]
        .isna()
        .sum()
    ),

    "missing_customer_id": (
        df["customer_id"]
        .isna()
        .sum()
    ),

    "missing_invoice_date": (
        df["invoice_date"]
        .isna()
        .sum()
    ),

    "missing_due_date": (
        df["due_date"]
        .isna()
        .sum()
    ),

    "missing_settled_date": (
        df["settled_date"]
        .isna()
        .sum()
    ),

    "missing_invoice_amount": (
        df["invoice_amount"]
        .isna()
        .sum()
    ),

    "invoice_amount_le_zero": (
        df["invoice_amount"] <= 0
    ).sum(),

    "due_before_invoice": (
        df["due_date"]
        < df["invoice_date"]
    ).sum(),

    "settled_before_invoice": (
        df["settled_date"]
        < df["invoice_date"]
    ).sum(),
})

display(quality_checks)

assert df["invoice_id"].is_unique

assert df["invoice_date"].notna().all()
assert df["due_date"].notna().all()
assert df["settled_date"].notna().all()

assert (df["invoice_amount"] > 0).all()

assert not (
    df["due_date"] < df["invoice_date"]
).any()

assert not (
    df["settled_date"] < df["invoice_date"]
).any()

print("Validaciones básicas superadas.")

rows                      2466
duplicated_rows              0
duplicated_invoice_id        0
missing_invoice_id           0
missing_customer_id          0
missing_invoice_date         0
missing_due_date             0
missing_settled_date         0
missing_invoice_amount       0
invoice_amount_le_zero       0
due_before_invoice           0
settled_before_invoice       0
dtype: int64

Validaciones básicas superadas.


In [8]:
target_summary = (
    df["late_payment"]
    .value_counts()
    .sort_index()
    .rename_axis("late_payment")
    .to_frame("count")
)

target_summary["percentage"] = (
    target_summary["count"]
    / len(df)
    * 100
)

display(target_summary)

assert df["late_payment"].sum() == 877

print(
    f"Tasa de retraso: "
    f"{df['late_payment'].mean():.2%}"
)

,count,percentage
late_payment,,
0,1589,64.436334
1,877,35.563666


Tasa de retraso: 35.56%


In [9]:
df.columns

Index(['country_code', 'customer_id', 'paperless_date', 'invoice_id',
       'invoice_date', 'due_date', 'invoice_amount', 'disputed',
       'settled_date', 'paperless_bill', 'days_to_settle_original',
       'days_late_original', 'days_delay_signed', 'late_payment', 'days_late'],
      dtype='object')

In [10]:
df = (
    df
    .sort_values(
        [
            "customer_id",
            "invoice_date",
            "invoice_id"
        ]
    )
    .reset_index(drop=True)
)

display(
    df[
        [
            "customer_id",
            "invoice_id",
            "invoice_date",
            "due_date",
            "settled_date",
            "invoice_amount",
            "late_payment"
        ]
    ].head(10)
)

,customer_id,invoice_id,invoice_date,due_date,settled_date,invoice_amount,late_payment
0,0187-ERLSR,4037644863,2012-03-29,2012-04-28,2012-04-25,62.68,0
1,0187-ERLSR,9471530987,2012-05-15,2012-06-14,2012-05-28,77.19,0
2,0187-ERLSR,9744145268,2012-05-21,2012-06-20,2012-06-04,51.65,0
3,0187-ERLSR,7214076449,2012-06-16,2012-07-16,2012-07-04,64.47,0
4,0187-ERLSR,1756742390,2012-09-05,2012-10-05,2012-09-14,84.57,0
5,0187-ERLSR,4063317759,2012-09-22,2012-10-22,2012-10-11,65.26,0
6,0187-ERLSR,4910161169,2012-12-06,2013-01-05,2012-12-19,59.00,0
7,0187-ERLSR,4160638076,2013-02-16,2013-03-18,2013-03-02,56.50,0
8,0187-ERLSR,5995302563,2013-03-04,2013-04-03,2013-03-11,31.72,0
9,0187-ERLSR,8350497297,2013-03-16,2013-04-15,2013-04-04,73.27,0


## 2. Separación entre información predictiva e información de resultado

El dataframe de trabajo contiene algunas variables relacionadas con el resultado final del pago porque son necesarias para reconstruir el comportamiento histórico de los clientes.

Sin embargo, estas variables nunca podrán utilizarse como predictores de la factura actual.

### Variables de resultado

- `settled_date`
- `days_to_settle_original`
- `days_late_original`
- `days_delay_signed`
- `days_late`
- `late_payment`

`late_payment` será exclusivamente la variable objetivo.

Las demás variables de resultado únicamente podrán utilizarse para construir características históricas cuando dicho resultado ya fuese conocido en la fecha de emisión de la factura que se desea predecir.

### Regla temporal

Para una factura emitida en una fecha `t`:

- factura anterior emitida: `invoice_date < t`;
- pago conocido: `settled_date < t`;
- factura abierta: `invoice_date < t` y `settled_date >= t`.

Esta regla se aplicará en todas las variables históricas para evitar data leakage.

## 3. Características de la factura actual

El primer grupo de variables se construye únicamente a partir de información conocida en el momento de emisión de la factura.

Estas variables no requieren consultar el comportamiento histórico del cliente y, por tanto, no presentan riesgo de fuga de información temporal.

Se utilizarán inicialmente:

- importe de la factura;
- transformación logarítmica del importe;
- modalidad de facturación;
- código de país;
- características temporales derivadas de la fecha de emisión.

El importe original se conservará para mantener su interpretación económica. También se calculará una transformación logarítmica que puede reducir el efecto de la asimetría de la distribución y facilitar el funcionamiento de determinados modelos lineales.

In [11]:
df["log_invoice_amount"] = np.log1p(
    df["invoice_amount"]
)

df[
    [
        "invoice_amount",
        "log_invoice_amount"
    ]
].describe()

,invoice_amount,log_invoice_amount
count,2466.000000,2466.000000
mean,59.895856,4.035343
std,20.435838,0.424226
min,5.260000,1.834180
25%,46.400000,3.858622
50%,60.560000,4.120012
75%,73.765000,4.314350
max,128.280000,4.861981


## Variables temporales

In [12]:
# Año
df["invoice_year"] = (
    df["invoice_date"].dt.year
)

# Mes
df["invoice_month"] = (
    df["invoice_date"].dt.month
)

# Trimestre
df["invoice_quarter"] = (
    df["invoice_date"].dt.quarter
)

# Día
df["invoice_weekday"] = (
    df["invoice_date"].dt.weekday
)

# Principio de mes
df["is_month_start"] = (
    df["invoice_date"]
    .dt.is_month_start
    .astype("int8")
)

# Final de mes
df["is_month_end"] = (
    df["invoice_date"]
    .dt.is_month_end
    .astype("int8")
)

## Tendencia temporal

In [13]:
dataset_start_date = (
    df["invoice_date"].min()
)

df["days_since_dataset_start"] = (
    df["invoice_date"]
    - dataset_start_date
).dt.days

display(
    df[
    [
        "invoice_date",
        "days_since_dataset_start"
    ]
].head()
)

df["days_since_dataset_start"].describe()

,invoice_date,days_since_dataset_start
0,2012-03-29,86
1,2012-05-15,133
2,2012-05-21,139
3,2012-06-16,165
4,2012-09-05,246


count    2466.000000
mean      351.622466
std       199.664637
min         0.000000
25%       181.000000
50%       354.000000
75%       519.000000
max       699.000000
Name: days_since_dataset_start, dtype: float64

## Check de nuevas variables

In [14]:
current_invoice_features = [
    "invoice_amount",
    "log_invoice_amount",
    "paperless_bill",
    "country_code",
    "invoice_year",
    "invoice_month",
    "invoice_quarter",
    "invoice_weekday",
    "is_month_start",
    "is_month_end",
    "days_since_dataset_start",
]

display(df[current_invoice_features].head(10))

df[current_invoice_features].dtypes

,invoice_amount,log_invoice_amount,paperless_bill,country_code,invoice_year,invoice_month,invoice_quarter,invoice_weekday,is_month_start,is_month_end,days_since_dataset_start
0,62.68,4.153871,PAPER,391,2012,3,1,3,0,0,86
1,77.19,4.359142,PAPER,391,2012,5,2,1,0,0,133
2,51.65,3.963666,PAPER,391,2012,5,2,0,0,0,139
3,64.47,4.181592,PAPER,391,2012,6,2,5,0,0,165
4,84.57,4.449335,PAPER,391,2012,9,3,2,0,0,246
5,65.26,4.193586,PAPER,391,2012,9,3,5,0,0,263
6,59.00,4.094345,PAPER,391,2012,12,4,3,0,0,338
7,56.50,4.051785,PAPER,391,2013,2,1,5,0,0,410
8,31.72,3.487987,PAPER,391,2013,3,1,0,0,0,426
9,73.27,4.307707,PAPER,391,2013,3,1,5,0,0,438


invoice_amount                     float64
log_invoice_amount                 float64
paperless_bill              string[python]
country_code                string[python]
invoice_year                         int32
invoice_month                        int32
invoice_quarter                      int32
invoice_weekday                      int32
is_month_start                        int8
is_month_end                          int8
days_since_dataset_start             int64
dtype: object

## Validaciones

In [15]:
feature_quality = pd.DataFrame({
    "dtype": df[current_invoice_features].dtypes.astype(str),
    "missing": df[current_invoice_features].isna().sum(),
    "nunique": df[current_invoice_features].nunique(),
})

feature_quality

,dtype,missing,nunique
invoice_amount,float64,0,2098
log_invoice_amount,float64,0,2098
paperless_bill,string,0,2
country_code,string,0,5
invoice_year,int32,0,2
invoice_month,int32,0,12
invoice_quarter,int32,0,4
invoice_weekday,int32,0,7
is_month_start,int8,0,2
is_month_end,int8,0,2


## Distribución de variables temporales

In [16]:
for col in [
    "invoice_year",
    "invoice_month",
    "invoice_quarter",
    "invoice_weekday",
    "is_month_start",
    "is_month_end",
]:
    print(f"\n--- {col} ---")
    print(
        df[col]
        .value_counts()
        .sort_index()
    )


--- invoice_year ---
invoice_year
2012    1277
2013    1189
Name: count, dtype: int64

--- invoice_month ---
invoice_month
1     201
2     197
3     223
4     209
5     237
6     197
7     216
8     205
9     240
10    202
11    217
12    122
Name: count, dtype: int64

--- invoice_quarter ---
invoice_quarter
1    621
2    643
3    661
4    541
Name: count, dtype: int64

--- invoice_weekday ---
invoice_weekday
0    347
1    342
2    365
3    336
4    352
5    378
6    346
Name: count, dtype: int64

--- is_month_start ---
is_month_start
0    2377
1      89
Name: count, dtype: int64

--- is_month_end ---
is_month_end
0    2379
1      87
Name: count, dtype: int64


### Codificación de variables categóricas

En esta fase se mantendrán las variables categóricas en su representación original.

Por ejemplo:

- `paperless_bill`: PAPER / ELECTRONIC;
- `country_code`: código categórico;
- `invoice_month`: 1–12;
- `invoice_weekday`: 0–6.

Todavía no se aplicará One-Hot Encoding, escalado ni otras transformaciones específicas de algoritmos.

Estas operaciones se realizarán posteriormente dentro del pipeline de modelado para evitar que el dataset Gold quede ligado a un modelo concreto y para garantizar que las transformaciones se ajusten únicamente utilizando los datos de entrenamiento.

## 4. Histórico conocido del cliente (`as-of`)

Una de las principales hipótesis del proyecto es que el comportamiento histórico de un cliente puede ayudar a predecir el riesgo de retraso de una nueva factura.

Sin embargo, estas variables deben construirse respetando estrictamente qué información estaba disponible en el momento de emisión.

Para una factura emitida en una fecha `t` se utilizarán tres conjuntos diferentes de información:

### Facturas anteriores emitidas

Se consideran únicamente facturas con:

`invoice_date < t`

Estas facturas permiten conocer el histórico comercial del cliente, por ejemplo:

- número de facturas anteriores;
- importe medio histórico;
- importe mediano;
- tiempo desde la factura anterior.

### Pagos históricos conocidos

Para conocer el comportamiento de pago solo pueden utilizarse facturas anteriores que ya hubiesen sido liquidadas antes de `t`:

`invoice_date < t`  
`settled_date < t`

A partir de ellas se calcularán variables como:

- número de pagos conocidos;
- tasa histórica de retraso;
- retraso medio, mediano y máximo;
- retraso del último pago;
- comportamiento reciente.

### Facturas abiertas

Una factura anterior se considera abierta cuando:

`invoice_date < t`  
`settled_date >= t`

Estas facturas ya existían en la fecha de emisión de la factura actual, pero todavía no se conocía su resultado final.

Permiten medir la exposición actual del cliente mediante variables como:

- número de facturas abiertas;
- importe pendiente;
- número de facturas vencidas;
- importe vencido;
- antigüedad de la deuda vencida.

Se utilizará una comparación temporal estricta (`< t`) para evitar asumir un orden de eventos dentro de un mismo día, ya que el dataset no contiene información horaria.

In [17]:
def build_customer_asof_features(data):
    """
    Construye variables históricas de cliente utilizando únicamente
    información disponible en la fecha de emisión de cada factura.

    Para una factura emitida en t:
        - previous_issued: invoice_date < t
        - known_settled: invoice_date < t y settled_date < t
        - open_invoices: invoice_date < t y settled_date >= t
        - overdue_open: open_invoices con due_date < t

    Las facturas emitidas el mismo día no se consideran historia previa,
    ya que no conocemos el orden exacto de emisión dentro del día.
    """

    features = pd.DataFrame(index=data.index)

    # Inicialización de variables de conteo
    count_features = [
        "previous_issued_count",
        "previous_settled_count",
        "previous_late_count",
        "open_invoice_count_at_issue",
        "overdue_invoice_count_at_issue",
    ]

    for col in count_features:
        features[col] = 0

    # Variables que pueden no estar disponibles para clientes sin histórico
    numeric_features = [
        "previous_late_rate",
        "previous_avg_delay",
        "previous_median_delay",
        "previous_max_delay",
        "last_payment_delay",
        "days_since_last_payment",
        "last_3_late_rate",
        "last_3_avg_delay",
        "last_5_late_rate",
        "last_5_avg_delay",
        "previous_avg_invoice_amount",
        "previous_median_invoice_amount",
        "days_since_previous_invoice",
        "open_invoice_amount_at_issue",
        "overdue_amount_at_issue",
    ]

    for col in numeric_features:
        features[col] = np.nan
        
    features["oldest_overdue_days_at_issue"] = 0.0

    for customer_id, group in data.groupby(
        "customer_id",
        sort=False
    ):

        group = group.sort_values(
            ["invoice_date", "invoice_id"]
        )

        for idx, row in group.iterrows():

            t = row["invoice_date"]

            # ----------------------------------------
            # 1. Facturas emitidas antes de t
            # ----------------------------------------

            previous_issued = group[
                group["invoice_date"] < t
            ]

            # ----------------------------------------
            # 2. Facturas cuyo resultado ya era conocido
            # ----------------------------------------

            known_settled = previous_issued[
                previous_issued["settled_date"] < t
            ]

            # ----------------------------------------
            # 3. Facturas todavía abiertas en t
            # ----------------------------------------

            open_invoices = previous_issued[
                previous_issued["settled_date"].isna()
                | (previous_issued["settled_date"] >= t)
            ]

            # ----------------------------------------
            # 4. Facturas abiertas que ya estaban vencidas
            # ----------------------------------------

            overdue_open = open_invoices[
                open_invoices["due_date"] < t
            ]

            # ========================================
            # Histórico comercial
            # ========================================

            features.at[
                idx,
                "previous_issued_count"
            ] = len(previous_issued)

            if len(previous_issued) > 0:

                features.at[
                    idx,
                    "previous_avg_invoice_amount"
                ] = previous_issued[
                    "invoice_amount"
                ].mean()

                features.at[
                    idx,
                    "previous_median_invoice_amount"
                ] = previous_issued[
                    "invoice_amount"
                ].median()

                last_invoice_date = (
                    previous_issued["invoice_date"].max()
                )

                features.at[
                    idx,
                    "days_since_previous_invoice"
                ] = (
                    t - last_invoice_date
                ).days

            # ========================================
            # Histórico de pagos conocido
            # ========================================

            features.at[
                idx,
                "previous_settled_count"
            ] = len(known_settled)

            if len(known_settled) > 0:

                features.at[
                    idx,
                    "previous_late_count"
                ] = known_settled[
                    "late_payment"
                ].sum()

                features.at[
                    idx,
                    "previous_late_rate"
                ] = known_settled[
                    "late_payment"
                ].mean()

                features.at[
                    idx,
                    "previous_avg_delay"
                ] = known_settled[
                    "days_delay_signed"
                ].mean()

                features.at[
                    idx,
                    "previous_median_delay"
                ] = known_settled[
                    "days_delay_signed"
                ].median()

                features.at[
                    idx,
                    "previous_max_delay"
                ] = known_settled[
                    "days_delay_signed"
                ].max()

                # Ordenamos por fecha real de pago para obtener
                # el comportamiento conocido más reciente
                known_by_payment = known_settled.sort_values(
                    ["settled_date", "invoice_id"]
                )

                last_known_payment = (
                    known_by_payment.iloc[-1]
                )

                features.at[
                    idx,
                    "last_payment_delay"
                ] = last_known_payment[
                    "days_delay_signed"
                ]

                features.at[
                    idx,
                    "days_since_last_payment"
                ] = (
                    t
                    - last_known_payment["settled_date"]
                ).days

                # ------------------------------------
                # Comportamiento reciente: últimas 3
                # ------------------------------------

                if len(known_by_payment) >= 3:

                    last_3 = known_by_payment.tail(3)

                    features.at[
                        idx,
                        "last_3_late_rate"
                    ] = last_3[
                        "late_payment"
                    ].mean()

                    features.at[
                        idx,
                        "last_3_avg_delay"
                    ] = last_3[
                        "days_delay_signed"
                    ].mean()

                # ------------------------------------
                # Comportamiento reciente: últimas 5
                # ------------------------------------

                if len(known_by_payment) >= 5:

                    last_5 = known_by_payment.tail(5)

                    features.at[
                        idx,
                        "last_5_late_rate"
                    ] = last_5[
                        "late_payment"
                    ].mean()

                    features.at[
                        idx,
                        "last_5_avg_delay"
                    ] = last_5[
                        "days_delay_signed"
                    ].mean()

            # ========================================
            # Exposición abierta
            # ========================================

            features.at[
                idx,
                "open_invoice_count_at_issue"
            ] = len(open_invoices)

            features.at[
                idx,
                "open_invoice_amount_at_issue"
            ] = open_invoices[
                "invoice_amount"
            ].sum()

            # ========================================
            # Deuda vencida
            # ========================================

            features.at[
                idx,
                "overdue_invoice_count_at_issue"
            ] = len(overdue_open)

            features.at[
                idx,
                "overdue_amount_at_issue"
            ] = overdue_open[
                "invoice_amount"
            ].sum()

            if len(overdue_open) > 0:

                overdue_days = (
                    t - overdue_open["due_date"]
                ).dt.days

                features.at[
                    idx,
                    "oldest_overdue_days_at_issue"
                ] = overdue_days.max()

    return features

In [18]:
asof_features = build_customer_asof_features(df)

print(
    f"Filas generadas: {len(asof_features):,}"
)

print(
    f"Variables generadas: "
    f"{asof_features.shape[1]}"
)

display(
    asof_features.head(10)
)

Filas generadas: 2,466
Variables generadas: 21


,previous_issued_count,previous_settled_count,previous_late_count,open_invoice_count_at_issue,overdue_invoice_count_at_issue,previous_late_rate,previous_avg_delay,previous_median_delay,previous_max_delay,last_payment_delay,...,last_3_late_rate,last_3_avg_delay,last_5_late_rate,last_5_avg_delay,previous_avg_invoice_amount,previous_median_invoice_amount,days_since_previous_invoice,open_invoice_amount_at_issue,overdue_amount_at_issue,oldest_overdue_days_at_issue
0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.0,0.0
1,1,1,0,0,0,0.0,-3.000000,-3.0,-3.0,-3.0,...,NaN,NaN,NaN,NaN,62.680000,62.680,47.0,0.00,0.0,0.0
2,2,1,0,1,0,0.0,-3.000000,-3.0,-3.0,-3.0,...,NaN,NaN,NaN,NaN,69.935000,69.935,6.0,77.19,0.0,0.0
3,3,3,0,0,0,0.0,-12.000000,-16.0,-3.0,-16.0,...,0.0,-12.000000,NaN,NaN,63.840000,62.680,26.0,0.00,0.0,0.0
4,4,4,0,0,0,0.0,-12.000000,-14.0,-3.0,-12.0,...,0.0,-15.000000,NaN,NaN,63.997500,63.575,81.0,0.00,0.0,0.0
5,5,5,0,0,0,0.0,-13.800000,-16.0,-3.0,-21.0,...,0.0,-16.333333,0.0,-13.8,68.112000,64.470,17.0,0.00,0.0,0.0
6,6,6,0,0,0,0.0,-13.333333,-14.0,-3.0,-11.0,...,0.0,-14.666667,0.0,-15.4,67.636667,64.865,75.0,0.00,0.0,0.0
7,7,7,0,0,0,0.0,-13.857143,-16.0,-3.0,-17.0,...,0.0,-16.333333,0.0,-15.4,66.402857,64.470,72.0,0.00,0.0,0.0
8,8,8,0,0,0,0.0,-14.125000,-16.0,-3.0,-16.0,...,0.0,-14.666667,0.0,-15.4,65.165000,63.575,16.0,0.00,0.0,0.0
9,9,9,0,0,0,0.0,-15.111111,-16.0,-3.0,-23.0,...,0.0,-18.666667,0.0,-17.6,61.448889,62.680,12.0,0.00,0.0,0.0


### Incorporarlas al dataframe

In [19]:
df = pd.concat(
    [
        df,
        asof_features
    ],
    axis=1
)

print(
    f"Dimensiones actuales del dataframe: "
    f"{df.shape}"
)

Dimensiones actuales del dataframe: (2466, 44)


### Interpretación de las variables históricas

Las variables construidas se dividen en tres bloques conceptualmente diferentes.

#### Histórico comercial

Estas variables describen la relación comercial anterior con el cliente:

- `previous_issued_count`
- `previous_avg_invoice_amount`
- `previous_median_invoice_amount`
- `days_since_previous_invoice`

Para calcularlas es suficiente con que las facturas anteriores ya hubiesen sido emitidas, ya que su importe y fecha eran conocidos independientemente de si habían sido cobradas.

#### Histórico de pagos

Estas variables describen únicamente resultados de pago que ya se conocían en la fecha de emisión:

- `previous_settled_count`
- `previous_late_count`
- `previous_late_rate`
- `previous_avg_delay`
- `previous_median_delay`
- `previous_max_delay`
- `last_payment_delay`
- `days_since_last_payment`
- `last_3_late_rate`
- `last_3_avg_delay`
- `last_5_late_rate`
- `last_5_avg_delay`

Una factura anterior que todavía estuviese abierta no interviene en estas métricas, porque su resultado futuro todavía era desconocido.

#### Exposición pendiente

Estas variables representan compromisos de pago ya existentes en la fecha de emisión:

- `open_invoice_count_at_issue`
- `open_invoice_amount_at_issue`
- `overdue_invoice_count_at_issue`
- `overdue_amount_at_issue`
- `oldest_overdue_days_at_issue`

Estas características permiten distinguir, por ejemplo, entre un cliente sin deuda pendiente y otro al que se emite una nueva factura mientras mantiene varias facturas anteriores vencidas.

### Validaciones del `as-of`

In [20]:
asof_checks = pd.Series({

    "settled_gt_issued": (
        df["previous_settled_count"]
        > df["previous_issued_count"]
    ).sum(),

    "late_gt_settled": (
        df["previous_late_count"]
        > df["previous_settled_count"]
    ).sum(),

    "overdue_gt_open": (
        df["overdue_invoice_count_at_issue"]
        > df["open_invoice_count_at_issue"]
    ).sum(),

    "overdue_amount_gt_open_amount": (
        df["overdue_amount_at_issue"]
        > df["open_invoice_amount_at_issue"]
    ).sum(),

    "late_rate_below_zero": (
        df["previous_late_rate"] < 0
    ).sum(),

    "late_rate_above_one": (
        df["previous_late_rate"] > 1
    ).sum(),

    "negative_open_count": (
        df["open_invoice_count_at_issue"] < 0
    ).sum(),

    "negative_overdue_count": (
        df["overdue_invoice_count_at_issue"] < 0
    ).sum(),
})

asof_checks

settled_gt_issued                0
late_gt_settled                  0
overdue_gt_open                  0
overdue_amount_gt_open_amount    0
late_rate_below_zero             0
late_rate_above_one              0
negative_open_count              0
negative_overdue_count           0
dtype: int64

In [21]:
assert (
    df.loc[
        df["overdue_invoice_count_at_issue"] == 0,
        "oldest_overdue_days_at_issue"
    ] == 0
).all()

assert (
    df.loc[
        df["overdue_invoice_count_at_issue"] > 0,
        "oldest_overdue_days_at_issue"
    ] > 0
).all()

print(
    "✓ La antigüedad de deuda vencida "
    "es coherente con el número de facturas vencidas."
)

✓ La antigüedad de deuda vencida es coherente con el número de facturas vencidas.


In [22]:
history_balance_error = (
    df["previous_issued_count"]
    != (
        df["previous_settled_count"]
        + df["open_invoice_count_at_issue"]
    )
)

print(
    "Filas que no cumplen la relación:",
    history_balance_error.sum()
)

assert history_balance_error.sum() == 0

print(
    "✓ La reconciliación del histórico es correcta."
)

Filas que no cumplen la relación: 0
✓ La reconciliación del histórico es correcta.


### Disponibilidad del historico

In [23]:
history_availability = pd.Series({

    "sin_historial_emitido": (
        df["previous_issued_count"] == 0
    ).mean(),

    "con_historial_emitido": (
        df["previous_issued_count"] >= 1
    ).mean(),

    "con_al_menos_1_pago_conocido": (
        df["previous_settled_count"] >= 1
    ).mean(),

    "con_al_menos_3_pagos_conocidos": (
        df["previous_settled_count"] >= 3
    ).mean(),

    "con_al_menos_5_pagos_conocidos": (
        df["previous_settled_count"] >= 5
    ).mean(),

    "con_facturas_abiertas": (
        df["open_invoice_count_at_issue"] > 0
    ).mean(),

    "con_facturas_vencidas": (
        df["overdue_invoice_count_at_issue"] > 0
    ).mean(),
}) * 100

history_availability.round(2)

sin_historial_emitido              4.06
con_historial_emitido             95.94
con_al_menos_1_pago_conocido      92.13
con_al_menos_3_pagos_conocidos    83.74
con_al_menos_5_pagos_conocidos    75.63
con_facturas_abiertas             57.95
con_facturas_vencidas              9.29
dtype: float64

### Distribucion de las features

In [24]:
historical_features = [
    "previous_issued_count",
    "previous_settled_count",
    "previous_late_count",
    "previous_late_rate",
    "previous_avg_delay",
    "previous_median_delay",
    "previous_max_delay",
    "last_payment_delay",
    "days_since_last_payment",
    "last_3_late_rate",
    "last_3_avg_delay",
    "last_5_late_rate",
    "last_5_avg_delay",
    "previous_avg_invoice_amount",
    "previous_median_invoice_amount",
    "days_since_previous_invoice",
    "open_invoice_count_at_issue",
    "open_invoice_amount_at_issue",
    "overdue_invoice_count_at_issue",
    "overdue_amount_at_issue",
    "oldest_overdue_days_at_issue",
]

df[
    historical_features
].describe().T

,count,mean,std,min,25%,50%,75%,max
previous_issued_count,2466.0,12.270073,7.879213,0.000000,6.000000,12.000000,18.000000,35.000000
previous_settled_count,2466.0,11.358070,7.837415,0.000000,5.000000,11.000000,17.000000,34.000000
previous_late_count,2466.0,4.195864,5.239126,0.000000,0.000000,2.000000,6.000000,30.000000
previous_late_rate,2272.0,0.381475,0.365167,0.000000,0.000000,0.272727,0.700000,1.000000
previous_avg_delay,2272.0,-2.272817,10.053774,-28.000000,-9.000000,-2.409091,5.500000,31.000000
previous_median_delay,2272.0,-2.800396,10.196954,-28.000000,-9.500000,-3.000000,5.000000,31.000000
previous_max_delay,2272.0,8.949384,12.786551,-28.000000,0.000000,8.000000,18.000000,45.000000
last_payment_delay,2272.0,-3.364437,12.478255,-30.000000,-12.000000,-4.000000,5.000000,45.000000
days_since_last_payment,2272.0,27.566901,26.669907,1.000000,8.000000,19.000000,39.000000,183.000000
last_3_late_rate,2065.0,0.358515,0.392395,0.000000,0.000000,0.333333,0.666667,1.000000


### Valores ausentes en las variables históricas

La aparición de valores `NaN` en algunas variables históricas no implica necesariamente un problema de calidad.

Por ejemplo, si una factura pertenece a un cliente que todavía no tenía ningún pago histórico conocido, no es posible calcular:

- `previous_late_rate`;
- `previous_avg_delay`;
- `previous_median_delay`;
- `previous_max_delay`;
- `last_payment_delay`;
- `days_since_last_payment`.

Del mismo modo, `last_3_late_rate` solo estará disponible cuando existan al menos tres pagos anteriores conocidos, y `last_5_late_rate` requerirá al menos cinco.

Estos valores ausentes tienen por tanto un significado empresarial: **ausencia de histórico suficiente**.

No se imputarán todavía.

La estrategia de imputación se definirá posteriormente dentro del pipeline de modelado y deberá ajustarse exclusivamente utilizando el conjunto de entrenamiento para evitar fuga de información.

Para que el modelo pueda distinguir explícitamente los casos sin histórico se crearán también indicadores binarios.

### Indicadores de disponibilidad del historico

In [25]:
df["is_new_customer"] = (
    df["previous_issued_count"] == 0
).astype("int8")

df["has_payment_history"] = (
    df["previous_settled_count"] > 0
).astype("int8")

df["has_3_payment_history"] = (
    df["previous_settled_count"] >= 3
).astype("int8")

df["has_5_payment_history"] = (
    df["previous_settled_count"] >= 5
).astype("int8")

df["has_open_invoices"] = (
    df["open_invoice_count_at_issue"] > 0
).astype("int8")

df["has_overdue_invoices"] = (
    df["overdue_invoice_count_at_issue"] > 0
).astype("int8")

df[
    [
        "is_new_customer",
        "has_payment_history",
        "has_3_payment_history",
        "has_5_payment_history",
        "has_open_invoices",
        "has_overdue_invoices",
    ]
].mean().mul(100).round(2)

is_new_customer           4.06
has_payment_history      92.13
has_3_payment_history    83.74
has_5_payment_history    75.63
has_open_invoices        57.95
has_overdue_invoices      9.29
dtype: float64

### Variables relativas al importe

In [26]:
df["amount_vs_previous_avg"] = (
    df["invoice_amount"]
    / df["previous_avg_invoice_amount"]
)

df["amount_vs_previous_median"] = (
    df["invoice_amount"]
    / df["previous_median_invoice_amount"]
)

df["amount_diff_vs_previous_avg"] = (
    df["invoice_amount"]
    - df["previous_avg_invoice_amount"]
)

### Variables relativas a exposicion

In [27]:
df["open_amount_vs_current_invoice"] = (
    df["open_invoice_amount_at_issue"]
    / df["invoice_amount"]
)

df["overdue_amount_vs_current_invoice"] = (
    df["overdue_amount_at_issue"]
    / df["invoice_amount"]
)

### Validacion de ratios

In [28]:
ratio_features = [
    "amount_vs_previous_avg",
    "amount_vs_previous_median",
    "amount_diff_vs_previous_avg",
    "open_amount_vs_current_invoice",
    "overdue_amount_vs_current_invoice",
]

display(df[ratio_features].describe().T)

infinite_values = pd.Series({
    col: np.isinf(
        df[col].dropna()
    ).sum()
    for col in ratio_features
})

infinite_values

,count,mean,std,min,25%,50%,75%,max
amount_vs_previous_avg,2366.0,1.019947,0.342548,0.152082,0.823387,0.991689,1.189093,5.269242
amount_vs_previous_median,2366.0,1.029893,0.369762,0.155852,0.820949,0.996551,1.199526,5.688634
amount_diff_vs_previous_avg,2366.0,0.206918,16.300551,-59.470000,-10.524524,-0.439808,10.965937,65.210000
open_amount_vs_current_invoice,2466.0,1.036414,1.418744,0.000000,0.000000,0.768096,1.545606,18.425876
overdue_amount_vs_current_invoice,2466.0,0.128169,0.505891,0.000000,0.000000,0.000000,0.000000,9.970284


amount_vs_previous_avg               0
amount_vs_previous_median            0
amount_diff_vs_previous_avg          0
open_amount_vs_current_invoice       0
overdue_amount_vs_current_invoice    0
dtype: int64

## Estado de las variables construidas

Hasta este punto se han generado características pertenecientes a cuatro grandes grupos:

### Factura actual

- `invoice_amount`
- `log_invoice_amount`
- `paperless_bill`
- `country_code`

### Variables temporales

- `invoice_year`
- `invoice_month`
- `invoice_quarter`
- `invoice_weekday`
- `is_month_start`
- `is_month_end`
- `days_since_dataset_start`

### Comportamiento histórico del cliente

- `previous_issued_count`
- `previous_settled_count`
- `previous_late_count`
- `previous_late_rate`
- `previous_avg_delay`
- `previous_median_delay`
- `previous_max_delay`
- `last_payment_delay`
- `days_since_last_payment`
- `last_3_late_rate`
- `last_3_avg_delay`
- `last_5_late_rate`
- `last_5_avg_delay`

### Histórico comercial

- `previous_avg_invoice_amount`
- `previous_median_invoice_amount`
- `days_since_previous_invoice`
- `amount_vs_previous_avg`
- `amount_vs_previous_median`
- `amount_diff_vs_previous_avg`

### Exposición pendiente

- `open_invoice_count_at_issue`
- `open_invoice_amount_at_issue`
- `overdue_invoice_count_at_issue`
- `overdue_amount_at_issue`
- `oldest_overdue_days_at_issue`
- `open_amount_vs_current_invoice`
- `overdue_amount_vs_current_invoice`

### Indicadores de disponibilidad

- `is_new_customer`
- `has_payment_history`
- `has_3_payment_history`
- `has_5_payment_history`
- `has_open_invoices`
- `has_overdue_invoices`

Todas estas variables se han construido utilizando exclusivamente información que podía estar disponible en la fecha de emisión de cada factura.

## 5. Auditoría del dataset candidato a Gold

Antes de construir el dataset definitivo para modelado se realizará una auditoría de las variables generadas.

El objetivo es comprobar:

- que no existan variables constantes o sin información;
- que los valores ausentes tengan una explicación coherente;
- que no se hayan generado valores infinitos;
- que ninguna variable de resultado futuro forme parte de los predictores;
- que los identificadores queden separados de las variables predictivas;
- que no existan columnas redundantes que aporten exactamente la misma información;
- que cada variable tenga un tipo de dato adecuado.

El dataset Gold contendrá tres grupos de columnas:

1. **Identificadores y contexto temporal**
   - necesarios para trazabilidad y división temporal;
   - no se utilizarán directamente como predictores.

2. **Features (`X`)**
   - información disponible en el momento de emisión.

3. **Target (`y`)**
   - `late_payment`.

Las columnas utilizadas únicamente para reconstruir el pasado no formarán parte de `X`.

### Definir grupos de columnas

In [29]:
identifier_columns = [
    "invoice_id",
    "customer_id",
    "invoice_date",
    "due_date",
]

candidate_features = [
    # -----------------------------
    # Factura actual
    # -----------------------------
    "invoice_amount",
    "log_invoice_amount",
    "paperless_bill",
    "country_code",

    # -----------------------------
    # Tiempo
    # -----------------------------
    "invoice_year",
    "invoice_month",
    "invoice_quarter",
    "invoice_weekday",
    "is_month_start",
    "is_month_end",
    "days_since_dataset_start",

    # -----------------------------
    # Histórico comercial
    # -----------------------------
    "previous_issued_count",
    "previous_avg_invoice_amount",
    "previous_median_invoice_amount",
    "days_since_previous_invoice",

    # -----------------------------
    # Histórico de pagos conocido
    # -----------------------------
    "previous_settled_count",
    "previous_late_count",
    "previous_late_rate",
    "previous_avg_delay",
    "previous_median_delay",
    "previous_max_delay",
    "last_payment_delay",
    "days_since_last_payment",

    # -----------------------------
    # Comportamiento reciente
    # -----------------------------
    "last_3_late_rate",
    "last_3_avg_delay",
    "last_5_late_rate",
    "last_5_avg_delay",

    # -----------------------------
    # Exposición pendiente
    # -----------------------------
    "open_invoice_count_at_issue",
    "open_invoice_amount_at_issue",
    "overdue_invoice_count_at_issue",
    "overdue_amount_at_issue",
    "oldest_overdue_days_at_issue",

    # -----------------------------
    # Ratios
    # -----------------------------
    "amount_vs_previous_avg",
    "amount_vs_previous_median",
    "amount_diff_vs_previous_avg",
    "open_amount_vs_current_invoice",
    "overdue_amount_vs_current_invoice",

    # -----------------------------
    # Disponibilidad de histórico
    # -----------------------------
    "is_new_customer",
    "has_payment_history",
    "has_3_payment_history",
    "has_5_payment_history",
    "has_open_invoices",
    "has_overdue_invoices",
]

# Target
target_column = "late_payment"

In [30]:
expected_columns = (
    identifier_columns
    + candidate_features
    + [target_column]
)

missing_expected_columns = [
    col
    for col in expected_columns
    if col not in df.columns
]

print(missing_expected_columns)

assert not missing_expected_columns, (
    f"Faltan columnas esperadas: "
    f"{missing_expected_columns}"
)

print("✓ Todas las columnas esperadas existen.")

[]
✓ Todas las columnas esperadas existen.


### Auditoría general de features

In [31]:
feature_audit = pd.DataFrame({
    "dtype": df[candidate_features].dtypes.astype(str),
    "missing_count": (
        df[candidate_features]
        .isna()
        .sum()
    ),
    "missing_pct": (
        df[candidate_features]
        .isna()
        .mean()
        .mul(100)
    ),
    "nunique": (
        df[candidate_features]
        .nunique(dropna=True)
    ),
})

feature_audit["unique_pct"] = (
    feature_audit["nunique"]
    / len(df)
    * 100
)

feature_audit.sort_values(
    "missing_pct",
    ascending=False
)

,dtype,missing_count,missing_pct,nunique,unique_pct
last_5_avg_delay,float64,601,24.371452,239,9.691809
last_5_late_rate,float64,601,24.371452,6,0.243309
last_3_avg_delay,float64,401,16.261152,156,6.326034
last_3_late_rate,float64,401,16.261152,4,0.162206
previous_late_rate,float64,194,7.866991,196,7.948094
days_since_last_payment,float64,194,7.866991,133,5.393350
previous_max_delay,float64,194,7.866991,60,2.433090
previous_median_delay,float64,194,7.866991,101,4.095702
previous_avg_delay,float64,194,7.866991,968,39.253852
last_payment_delay,float64,194,7.866991,66,2.676399


### Variables constantes o casi constantes

In [32]:
constant_features = (
    feature_audit[
        feature_audit["nunique"] <= 1
    ]
)

print(constant_features)

binary_features = [
    col
    for col in candidate_features
    if df[col].nunique(dropna=True) == 2
]

for col in binary_features:
    print(f"\n--- {col} ---")
    print(
        df[col]
        .value_counts(
            normalize=True,
            dropna=False
        )
        .mul(100)
        .round(2)
    )

Empty DataFrame
Columns: [dtype, missing_count, missing_pct, nunique, unique_pct]
Index: []

--- paperless_bill ---
paperless_bill
PAPER         51.22
ELECTRONIC    48.78
Name: proportion, dtype: Float64

--- invoice_year ---
invoice_year
2012    51.78
2013    48.22
Name: proportion, dtype: float64

--- is_month_start ---
is_month_start
0    96.39
1     3.61
Name: proportion, dtype: float64

--- is_month_end ---
is_month_end
0    96.47
1     3.53
Name: proportion, dtype: float64

--- is_new_customer ---
is_new_customer
0    95.94
1     4.06
Name: proportion, dtype: float64

--- has_payment_history ---
has_payment_history
1    92.13
0     7.87
Name: proportion, dtype: float64

--- has_3_payment_history ---
has_3_payment_history
1    83.74
0    16.26
Name: proportion, dtype: float64

--- has_5_payment_history ---
has_5_payment_history
1    75.63
0    24.37
Name: proportion, dtype: float64

--- has_open_invoices ---
has_open_invoices
1    57.95
0    42.05
Name: proportion, dtype: float64


### Variables constantes y de baja variabilidad

Una variable constante no puede ayudar a distinguir entre facturas y, por tanto, no aporta información al modelo.

Estas variables serán eliminadas de la lista final de predictores.

Las variables binarias con una categoría poco frecuente no se eliminarán automáticamente. Su utilidad se evaluará posteriormente durante el modelado, siempre que exista un número suficiente de observaciones en ambas categorías.

### Eliminar solo variables constantes

In [33]:
constant_feature_names = (
    feature_audit[
        feature_audit["nunique"] <= 1
    ]
    .index
    .tolist()
)

print(
    "Variables constantes:",
    constant_feature_names
)

model_features = [
    col
    for col in candidate_features
    if col not in constant_feature_names
]

print(
    f"Features candidatas iniciales: "
    f"{len(candidate_features)}"
)

print(
    f"Features después de eliminar constantes: "
    f"{len(model_features)}"
)

Variables constantes: []
Features candidatas iniciales: 43
Features después de eliminar constantes: 43


### Comprobamos infinitos

In [34]:
numeric_model_features = (
    df[model_features]
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

infinite_summary = pd.Series({
    col: np.isinf(
        df[col].dropna()
    ).sum()
    for col in numeric_model_features
})

infinite_summary[
    infinite_summary > 0
]

assert infinite_summary.sum() == 0

print("✓ No existen valores infinitos.")

✓ No existen valores infinitos.


### Auditoria explicita de leakage

In [35]:
forbidden_predictors = [
    "settled_date",
    "days_to_settle_original",
    "days_late_original",
    "days_delay_signed",
    "days_late",
    "late_payment",
    "paperless_date",
    "disputed",
]

leakage_found = [
    col
    for col in forbidden_predictors
    if col in model_features
]

print(leakage_found)

assert not leakage_found, (
    f"Se han encontrado variables prohibidas "
    f"en X: {leakage_found}"
)

print(
    "✓ No se detectan variables prohibidas "
    "en las features del modelo."
)

[]
✓ No se detectan variables prohibidas en las features del modelo.


### Comprobar identificadores

In [36]:
identifier_leakage = [
    col
    for col in [
        "invoice_id",
        "customer_id",
        "invoice_date",
        "due_date",
    ]
    if col in model_features
]

identifier_leakage


[]

### Revisar redundancias

In [37]:
numeric_corr = (
    df[numeric_model_features]
    .corr()
    .abs()
)

high_corr_pairs = []

for i, col_1 in enumerate(numeric_corr.columns):
    for col_2 in numeric_corr.columns[i + 1:]:

        corr_value = numeric_corr.loc[
            col_1,
            col_2
        ]

        if corr_value >= 0.95:
            high_corr_pairs.append({
                "feature_1": col_1,
                "feature_2": col_2,
                "abs_correlation": corr_value,
            })

high_corr_df = (
    pd.DataFrame(high_corr_pairs)
    .sort_values(
        "abs_correlation",
        ascending=False
    )
    if high_corr_pairs
    else pd.DataFrame(
        columns=[
            "feature_1",
            "feature_2",
            "abs_correlation"
        ]
    )
)

high_corr_df

,feature_1,feature_2,abs_correlation
1,previous_issued_count,previous_settled_count,0.992098
3,previous_avg_delay,previous_median_delay,0.989446
2,previous_avg_invoice_amount,previous_median_invoice_amount,0.982189
8,amount_vs_previous_avg,amount_vs_previous_median,0.970308
0,invoice_month,invoice_quarter,0.970098
6,last_3_avg_delay,last_5_avg_delay,0.968334
4,previous_avg_delay,last_5_avg_delay,0.964392
5,previous_median_delay,last_5_avg_delay,0.952622
7,overdue_invoice_count_at_issue,has_overdue_invoices,0.951030


### Variables correlacionadas y redundancia

La existencia de una correlación elevada entre dos features no implica automáticamente que una deba eliminarse.

En esta fase se conservarán las variables candidatas salvo que sean:

- constantes;
- duplicados exactos;
- variables de resultado;
- identificadores;
- variables con fuga de información.

La selección definitiva dependerá también del tipo de modelo.

Los modelos lineales pueden verse más afectados por la multicolinealidad, mientras que modelos basados en árboles suelen tolerar mejor variables correlacionadas.

Por tanto, la posible eliminación de features altamente correlacionadas se evaluará durante la fase de modelado y no se realizará únicamente a partir de la matriz de correlaciones.

### Comprobar duplicados exactos

In [38]:
exact_duplicate_features = []

for i, col_1 in enumerate(model_features):

    for col_2 in model_features[i + 1:]:

        same = (
            df[col_1]
            .fillna("__MISSING__")
            .equals(
                df[col_2]
                .fillna("__MISSING__")
            )
        )

        if same:
            exact_duplicate_features.append(
                (col_1, col_2)
            )

exact_duplicate_features

[]

### Separar variables categoricas y numericas

In [39]:
categorical_features = [
    "paperless_bill",
    "country_code",
]

numeric_features = [
    col
    for col in model_features
    if col not in categorical_features
]

print(
    f"Variables numéricas: "
    f"{len(numeric_features)}"
)

print(
    f"Variables categóricas: "
    f"{len(categorical_features)}"
)

print("\nCategóricas:")
print(categorical_features)

Variables numéricas: 41
Variables categóricas: 2

Categóricas:
['paperless_bill', 'country_code']


### Tratamiento de valores ausentes

Los valores ausentes presentes en algunas features históricas son estructurales y no se consideran errores de calidad.

Por ejemplo:

- un cliente sin pagos anteriores no dispone de `previous_late_rate`;
- un cliente con menos de tres pagos conocidos no dispone de `last_3_late_rate`;
- un cliente con menos de cinco pagos conocidos no dispone de `last_5_late_rate`;
- un cliente sin facturas anteriores no dispone de estadísticas sobre su comportamiento histórico.

Estos valores representan ausencia de histórico suficiente y no se imputarán durante el feature engineering.

La imputación se realizará posteriormente dentro del pipeline de modelado utilizando exclusivamente información del conjunto de entrenamiento, evitando así que validation o test influyan en las transformaciones.

Para diferenciar la ausencia de histórico de valores reales iguales a cero se han creado variables indicadoras como:

- `is_new_customer`;
- `has_payment_history`;
- `has_3_payment_history`;
- `has_5_payment_history`.

En el caso de `oldest_overdue_days_at_issue`, la ausencia de facturas vencidas se representa explícitamente mediante el valor `0`.

En este caso, un valor cero tiene una interpretación empresarial clara: el cliente no presenta antigüedad de deuda vencida en el momento de emisión de la factura.

De esta forma se evita imputar posteriormente un valor artificial a clientes que realmente no tenían ninguna factura vencida.

### Auditoria final de missingness

In [40]:
missing_features = pd.DataFrame({
    "missing_count": (
        df[model_features]
        .isna()
        .sum()
    ),
    "missing_pct": (
        df[model_features]
        .isna()
        .mean()
        .mul(100)
    )
})

missing_features = (
    missing_features[
        missing_features["missing_count"] > 0
    ]
    .sort_values(
        "missing_pct",
        ascending=False
    )
)

missing_features

,missing_count,missing_pct
last_5_late_rate,601,24.371452
last_5_avg_delay,601,24.371452
last_3_late_rate,401,16.261152
last_3_avg_delay,401,16.261152
previous_late_rate,194,7.866991
previous_avg_delay,194,7.866991
previous_median_delay,194,7.866991
previous_max_delay,194,7.866991
last_payment_delay,194,7.866991
days_since_last_payment,194,7.866991


### Definir estructura Gold

In [41]:
gold_columns = (
    identifier_columns
    + model_features
    + [target_column]
)

gold_df = (
    df[gold_columns]
    .copy()
    .sort_values(
        ["invoice_date", "invoice_id"]
    )
    .reset_index(drop=True)
)

print(
    f"Gold shape: {gold_df.shape}"
)

display(
    gold_df.head()
)

Gold shape: (2466, 48)


,invoice_id,customer_id,invoice_date,due_date,invoice_amount,log_invoice_amount,paperless_bill,country_code,invoice_year,invoice_month,...,amount_diff_vs_previous_avg,open_amount_vs_current_invoice,overdue_amount_vs_current_invoice,is_new_customer,has_payment_history,has_3_payment_history,has_5_payment_history,has_open_invoices,has_overdue_invoices,late_payment
0,280670965,3993-QUNVJ,2012-01-03,2012-02-02,50.39,3.939444,PAPER,770,2012,1,...,NaN,0.0,0.0,1,0,0,0,0,0,0
1,5133177585,6708-DPYTF,2012-01-03,2012-02-02,55.37,4.031937,PAPER,391,2012,1,...,NaN,0.0,0.0,1,0,0,0,0,0,1
2,5928070131,1604-LIFKX,2012-01-03,2012-02-02,97.60,4.591071,PAPER,818,2012,1,...,NaN,0.0,0.0,1,0,0,0,0,0,1
3,6050714721,8887-NCUZC,2012-01-03,2012-02-02,15.99,2.832625,PAPER,818,2012,1,...,NaN,0.0,0.0,1,0,0,0,0,0,1
4,6393629835,5164-VMYWJ,2012-01-03,2012-02-02,71.33,4.281239,PAPER,406,2012,1,...,NaN,0.0,0.0,1,0,0,0,0,0,0


### Validacion de Gold

In [42]:
gold_checks = pd.Series({

    "rows": len(gold_df),

    "columns": gold_df.shape[1],

    "duplicated_rows": (
        gold_df.duplicated().sum()
    ),

    "duplicated_invoice_id": (
        gold_df["invoice_id"]
        .duplicated()
        .sum()
    ),

    "missing_invoice_id": (
        gold_df["invoice_id"]
        .isna()
        .sum()
    ),

    "missing_customer_id": (
        gold_df["customer_id"]
        .isna()
        .sum()
    ),

    "missing_invoice_date": (
        gold_df["invoice_date"]
        .isna()
        .sum()
    ),

    "target_missing": (
        gold_df[target_column]
        .isna()
        .sum()
    ),

    "target_positive": (
        gold_df[target_column]
        .sum()
    ),

    "target_positive_pct": (
        gold_df[target_column]
        .mean()
        * 100
    ),
})

gold_checks

rows                     2466.000000
columns                    48.000000
duplicated_rows             0.000000
duplicated_invoice_id       0.000000
missing_invoice_id          0.000000
missing_customer_id         0.000000
missing_invoice_date        0.000000
target_missing              0.000000
target_positive           877.000000
target_positive_pct        35.563666
dtype: float64

In [43]:
assert len(gold_df) == len(raw_df)

assert gold_df["invoice_id"].is_unique

assert gold_df["invoice_id"].notna().all()

assert gold_df["customer_id"].notna().all()

assert gold_df["invoice_date"].notna().all()

assert gold_df[target_column].notna().all()

assert set(
    gold_df[target_column].unique()
).issubset({0, 1})

assert gold_df[target_column].sum() == 877

print(
    "✓ Todas las validaciones del dataset Gold "
    "se han superado."
)

✓ Todas las validaciones del dataset Gold se han superado.


## 6. Definición final de la capa Gold

El dataset `gold_invoice_risk` constituye el resultado final del proceso de preparación y feature engineering para el modelo de riesgo de retraso.

### Granularidad

Una fila representa una factura.

### Identificadores

- `invoice_id`
- `customer_id`
- `invoice_date`
- `due_date`

Estas variables se mantienen por motivos de trazabilidad y división temporal, pero no se utilizarán directamente como predictores.

### Target

`late_payment`

- `0`: factura pagada antes o en su fecha de vencimiento;
- `1`: factura pagada después de su fecha de vencimiento.

### Features

Las variables predictivas describen:

- características de la factura actual;
- contexto temporal;
- histórico comercial del cliente;
- comportamiento histórico de pago conocido;
- comportamiento reciente;
- exposición pendiente;
- deuda vencida;
- importe relativo al comportamiento habitual;
- disponibilidad de histórico.

Todas las features han sido construidas utilizando únicamente información que podía estar disponible en la fecha de emisión de cada factura.

### Valores ausentes

Los valores ausentes de determinadas variables históricas representan falta de histórico suficiente y se conservarán en la capa Gold.

Su tratamiento se realizará posteriormente dentro del pipeline de modelado para evitar contaminación entre entrenamiento, validación y test.

### Variables excluidas

No se incluyen como predictores variables relacionadas con el resultado futuro del pago, identificadores puros ni variables cuya disponibilidad temporal no pueda garantizarse.

### Guardar Parquet

In [44]:
GOLD_FILE = (
    GOLD_DIR
    / "gold_invoice_risk.parquet"
)

gold_df.to_parquet(
    GOLD_FILE,
    index=False
)

print(
    f"Dataset Gold guardado en:\n"
    f"{GOLD_FILE.resolve()}"
)

Dataset Gold guardado en:
C:\Users\jesus\OneDrive\Documents\Evolve Data Science\portfolio\sistema_predictivo_tesoreria_y_riesgo_de_impago\data\gold\gold_invoice_risk.parquet


In [45]:
gold_check = pd.read_parquet(
    GOLD_FILE
)

print(
    f"Shape original: {gold_df.shape}"
)

print(
    f"Shape recargado: {gold_check.shape}"
)

assert gold_check.shape == gold_df.shape

assert (
    gold_check["invoice_id"]
    .tolist()
    ==
    gold_df["invoice_id"]
    .tolist()
)

assert (
    gold_check["late_payment"]
    .tolist()
    ==
    gold_df["late_payment"]
    .tolist()
)

print(
    "✓ El fichero Parquet se ha guardado "
    "y recuperado correctamente."
)

Shape original: (2466, 48)
Shape recargado: (2466, 48)
✓ El fichero Parquet se ha guardado y recuperado correctamente.


### Tabla final de roles

In [46]:
feature_roles = []

for col in gold_df.columns:

    if col in identifier_columns:
        role = "IDENTIFIER"

    elif col == target_column:
        role = "TARGET"

    elif col in categorical_features:
        role = "CATEGORICAL_FEATURE"

    else:
        role = "NUMERIC_FEATURE"

    feature_roles.append({
        "column": col,
        "role": role,
        "dtype": str(
            gold_df[col].dtype
        ),
        "missing_pct": (
            gold_df[col]
            .isna()
            .mean()
            * 100
        ),
        "nunique": (
            gold_df[col]
            .nunique(
                dropna=True
            )
        ),
    })

feature_dictionary = pd.DataFrame(
    feature_roles
)

feature_dictionary

,column,role,dtype,missing_pct,nunique
0,invoice_id,IDENTIFIER,string,0.000000,2466
1,customer_id,IDENTIFIER,string,0.000000,100
2,invoice_date,IDENTIFIER,datetime64[ns],0.000000,681
3,due_date,IDENTIFIER,datetime64[ns],0.000000,681
4,invoice_amount,NUMERIC_FEATURE,float64,0.000000,2098
5,log_invoice_amount,NUMERIC_FEATURE,float64,0.000000,2098
6,paperless_bill,CATEGORICAL_FEATURE,string,0.000000,2
7,country_code,CATEGORICAL_FEATURE,string,0.000000,5
8,invoice_year,NUMERIC_FEATURE,int32,0.000000,2
9,invoice_month,NUMERIC_FEATURE,int32,0.000000,12


# Conclusiones del Feature Engineering

El proceso de feature engineering ha transformado el dataset original de cuentas a cobrar en un dataset Gold preparado para las siguientes fases posteriores de modelado.

El dataset final mantiene una granularidad de una fila por factura y contiene 48 columnas:

- 4 variables de identificación y contexto temporal;
- 43 features candidatas;
- 1 variable objetivo (`late_payment`).

La variable objetivo conserva la distribución observada durante el EDA:

- 1.589 facturas pagadas de forma puntual o anticipada;
- 877 facturas pagadas después del vencimiento;
- tasa de retraso del 35,56 %.

### Construccion temporal de las variables

Las variables históricas se han construido utilizando lógica `as-of`, tomando como referencia la fecha de emisión de cada factura.

Para una factura emitida en una fecha `t`:

- solo se consideran anteriores las facturas con `invoice_date < t`;
- su comportamiento de pago únicamente se considera conocido si `settled_date < t`;
- una factura anterior se considera abierta cuando todavía no había sido liquidada en `t`;
- una factura se considera vencida únicamente cuando su fecha de vencimiento era anterior a `t`.

Las facturas emitidas o liquidadas en el mismo día no se utilizan como información previa, debido a que el dataset no proporciona información horaria que permita determinar el orden exacto de los eventos.

Las validaciones realizadas confirman que la reconstrucción temporal es consistente. En todas las observaciones se cumple:

`previous_issued_count = previous_settled_count + open_invoice_count_at_issue`

No se han detectado inconsistencias entre el número de facturas emitidas, liquidadas, abiertas o vencidas.


### Disponibilidad de historico

El dataset presenta una disponibilidad de histórico elevada:

- el 95,94 % de las facturas pertenece a clientes con al menos una factura anterior;
- el 92,13 % dispone de al menos un pago anterior cuyo resultado ya era conocido;
- el 83,74 % dispone de al menos tres pagos conocidos;
- el 75,63 % dispone de al menos cinco pagos conocidos.

Esto permite construir variables de comportamiento histórico para la mayoría de las observaciones.

Además, el 57,95 % de las facturas se emite mientras el cliente mantiene alguna factura anterior todavía abierta.

El 9,29 % se emite cuando el cliente ya presenta alguna factura vencida y pendiente de pago.

Estas situaciones permiten incorporar al modelo información relacionada con la exposición pendiente y la existencia de deuda vencida.


### Variables contruidas

Las features candidatas representan diferentes dimensiones de información:

- características de la factura actual;
- contexto temporal;
- histórico comercial del cliente;
- comportamiento histórico de pago;
- comportamiento reciente;
- facturas abiertas;
- deuda vencida;
- importe relativo al comportamiento habitual del cliente;
- disponibilidad de histórico.

La combinación de estas variables permite representar no solo las características de la factura que se quiere predecir, sino también la situación conocida del cliente en el momento de su emisión.

### Valores ausentes

Los valores ausentes presentes en determinadas variables históricas son principalmente estructurales.

Por ejemplo, no es posible calcular una tasa histórica de retraso cuando todavía no existe ningún pago anterior conocido.

De forma similar, las métricas correspondientes a las últimas tres o cinco operaciones requieren disponer de suficiente histórico.

Por esta razón, estos valores no se imputan durante el feature engineering.

La imputación se realizará posteriormente dentro del pipeline de modelado utilizando exclusivamente los datos de entrenamiento.

Se han creado además variables indicadoras que permiten distinguir explícitamente entre clientes con y sin histórico suficiente.

En `oldest_overdue_days_at_issue`, la ausencia de facturas vencidas se representa mediante el valor `0`, ya que en este caso existe una interpretación empresarial directa: el cliente no presenta deuda vencida en ese momento.

### Redundancia entre variables

No se han encontrado columnas exactamente duplicadas ni features constantes.

Se han identificado algunas variables fuertemente correlacionadas, entre ellas:

- número de facturas anteriores y número de pagos anteriores conocidos;
- media y mediana histórica del retraso;
- media y mediana histórica del importe;
- ratios construidos utilizando ambas medidas;
- mes y trimestre de emisión.

Estas variables se mantienen inicialmente en la capa Gold.

La posible reducción de redundancia se realizará durante la fase de modelado, teniendo en cuenta las características del algoritmo utilizado y su rendimiento sobre datos futuros.

La existencia de una correlación elevada no implica necesariamente que dos variables sean conceptualmente equivalentes.

### Prevención de fuga de información

Las comprobaciones realizadas confirman que ninguna de las variables identificadas como información futura está incluida entre las features del modelo.

Se han excluido como predictores:

- `settled_date`;
- `days_to_settle_original`;
- `days_late_original`;
- `days_delay_signed`;
- `days_late`;
- `paperless_date`;
- `disputed`;
- `late_payment`, excepto como variable objetivo.

Los identificadores:

- `invoice_id`;
- `customer_id`;
- `invoice_date`;
- `due_date`;

se conservan únicamente para trazabilidad, análisis de errores y división temporal y tampoco forman parte de las variables predictivas.

No se han detectado valores infinitos, identificadores incorporados accidentalmente a `X` ni columnas exactamente duplicadas.

### Validación de la capa Gold

El dataset final contiene:

- 2.466 registros;
- 48 columnas;
- 0 registros duplicados;
- 0 identificadores de factura duplicados;
- 0 identificadores principales ausentes;
- 0 valores ausentes en el target;
- 877 casos positivos;
- una tasa de retraso del 35,56 %.

El fichero `gold_invoice_risk.parquet` se ha guardado y leído de nuevo correctamente, manteniendo sus dimensiones, identificadores y variable objetivo.

### Decisiones que se trasladan al modelado

La capa Gold conserva un conjunto amplio de features candidatas.

Algunas variables presentan una correlación elevada y determinadas variables temporales pueden admitir diferentes formas de representación.

Por ejemplo:

- `invoice_month`;
- `invoice_quarter`;
- `invoice_weekday`.

Estas decisiones no se resolverán modificando la capa Gold.

Se evaluarán posteriormente dentro del pipeline de modelado, donde será posible comparar diferentes representaciones y seleccionar aquellas que mejor generalicen sobre datos futuros.

Del mismo modo, la imputación, el escalado y la codificación de variables categóricas se realizarán exclusivamente dentro de los pipelines de entrenamiento.

### Veredicto

La capa Gold se considera válida para comenzar la fase de modelado.

Las features construidas respetan el momento real de predicción y permiten estudiar si el comportamiento histórico del cliente, su exposición pendiente y las características de la factura aportan capacidad predictiva sobre el riesgo de retraso.

La siguiente fase utilizará exclusivamente `gold_invoice_risk.parquet` y realizará una división temporal entre entrenamiento, validación y test.

El primer objetivo será establecer un baseline trivial y posteriormente comprobar si una regresión logística es capaz de superar claramente dicha referencia antes de introducir modelos de mayor complejidad.